## Clean version of our assignment

Importing packages 

In [40]:
import pandas as pd
import os
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn import datasets as ds
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import decomposition

from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors

from sklearn.linear_model import LogisticRegression
from scipy.stats import loguniform, randint 
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import sklearn.metrics as sklm
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.tree import plot_tree
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report, average_precision_score, precision_recall_curve

Load data 

In [27]:
with zipfile.ZipFile("ecg_data.zip","r") as zip_ref:
    zip_ref.extractall("ecg_data")

def load_data():
    this_directory = os.getcwd()
    data = pd.read_csv(os.path.join(this_directory, 'ecg_data/ecg_data.csv'), index_col=0)
    return data

raw_data = load_data()

Data description

In [28]:
print(f'The number of samples: {len(raw_data.index)}')
print(f'The number of columns: {len(raw_data.columns)}')

print(f'The number of NaN values in the entire dataframe: {raw_data.isnull().sum().sum()}')
print(f'The number of samples with label 0: {len(raw_data[raw_data["label"] == 0])}')
print(f'The number of samples with label 1: {len(raw_data[raw_data["label"] == 1])}')
print(f'The percentage of samples with label 0 is thus {len(raw_data[raw_data["label"] == 0])/len(raw_data.index)*100:.2f}%', 
      f'and the percentage with label 1 {len(raw_data[raw_data["label"] == 1])/len(raw_data.index)*100:.2f}%')

# print(raw_data.groupby('label').count())
# print(raw_data.groupby('label').mean())
# print(raw_data.groupby('label').var())
# print(raw_data.groupby('label').std())


The number of samples: 827
The number of columns: 9001
The number of NaN values in the entire dataframe: 0
The number of samples with label 0: 681
The number of samples with label 1: 146
The percentage of samples with label 0 is thus 82.35% and the percentage with label 1 17.65%


The baseline for the AUC of the precision-recall curve is equal to the number of positives / the total, so in our case, it's 0.18

Splitting the data in training and test sets

In [36]:
X = raw_data.drop('label', axis=1)
Y = raw_data['label']

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=4, stratify=Y)

Logistic regression model with L1-regularisation to reduce the number of features 

In [42]:
lr_model = LogisticRegression(penalty='l1', solver='saga', class_weight='balanced', C=0.05, random_state=4) #saga solver requires features to be scaled for model conversion
lr_model.fit(x_train, y_train)  
y_pred = lr_model.predict(x_test)  
probabilities = lr_model.predict_proba(x_test)

print(f"CL Report:\n",classification_report(y_test, y_pred, zero_division=1))

precision = precision_score(y_test, y_pred)
print("Precision:", precision)

accuracy = sklm.accuracy_score(y_test, y_pred)
print('Accuracy:', round(accuracy,3))

f1 = f1_score(y_test, y_pred)
print("F1-Score:", f1)

recall = recall_score(y_test, y_pred)
print("Recall (Sensitivity):", recall)

confus_matr = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = confus_matr.ravel()
print("TN: {}, FP: {}, FN: {}, TP: {}\n".format(TN, FP, FN, TP))


c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


CL Report:
               precision    recall  f1-score   support

           0       0.91      0.88      0.90       137
           1       0.52      0.59      0.55        29

    accuracy                           0.83       166
   macro avg       0.71      0.73      0.72       166
weighted avg       0.84      0.83      0.84       166

Precision: 0.5151515151515151
Accuracy: 0.831
F1-Score: 0.5483870967741935
Recall (Sensitivity): 0.5862068965517241
TN: 121, FP: 16, FN: 12, TP: 17



c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Find the optimal hyperparameters with cross validation and a random search

In [ ]:
pipeline_regression = Pipeline([
    ('scaler', RobustScaler()),  #data scaling is important for model convergence with the saga solver
    ('classifier', LogisticRegression(
        solver='saga',
        random_state=4,
        max_iter=5000  # important for convergence
    ))
])

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=4)

param_dist = {
    'classifier__C': loguniform(1e-3, 10),  
    'classifier__penalty': ['l1', 'l2'],
    'classifier__class_weight': [None, 'balanced']
}

random_search = RandomizedSearchCV(
    pipeline_regression,
    param_distributions=param_dist,
    n_iter=30,  # number of random combinations
    cv=kf,
    scoring='average_precision',
    n_jobs=-1,
    random_state=4
)

random_search.fit(x_train, y_train)

print('Best parameters found:\n', random_search.best_params_)
print("Best score:", random_search.best_score_)

c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best parameters found:
 {'classifier__C': np.float64(0.09798880434744318), 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1'}
Best score: 0.5200864350631541


c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [33]:
param_grid_regression = {
    'classifier__C': [0.05, 0.1, 0.15],
    'classifier__penalty': ['l1'],  
    'classifier__class_weight': ['balanced'],  
}

grid_search_regression = GridSearchCV(pipeline_regression, param_grid_regression, cv=kf, scoring='average_precision', n_jobs=-1)
grid_search_regression.fit(x_train, y_train)

print('Best parameters found:\n', grid_search_regression.best_params_)
print("Beste score:", grid_search_regression.best_score_)

c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best parameters found:
 {'classifier__C': 0.05, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1'}
Beste score: 0.5380173600717999


c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Train logistic regression with optimal parameters 

In [43]:
lr_model = grid_search_regression.best_estimator_ 
y_pred_lr = lr_model.predict(x_test)  
probabilities_lr = lr_model.predict_proba(x_test)

print(f"CL Report:\n",classification_report(y_test, y_pred, zero_division=1))

precision = precision_score(y_test, y_pred)
print("Precision:", precision)

accuracy = sklm.accuracy_score(y_test, y_pred)
print('Accuracy:', round(accuracy,3))

f1 = f1_score(y_test, y_pred)
print("F1-Score:", f1)

recall = recall_score(y_test, y_pred)
print("Recall (Sensitivity):", recall)

ave_precision = average_precision_score(y_test, probabilities_lr[:, 1])
print("PR-AUC:", ave_precision)

confus_matr = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = confus_matr.ravel()
print("TN: {}, FP: {}, FN: {}, TP: {}\n".format(TN, FP, FN, TP))

CL Report:
               precision    recall  f1-score   support

           0       0.91      0.88      0.90       137
           1       0.52      0.59      0.55        29

    accuracy                           0.83       166
   macro avg       0.71      0.73      0.72       166
weighted avg       0.84      0.83      0.84       166

Precision: 0.5151515151515151
Accuracy: 0.831
F1-Score: 0.5483870967741935
Recall (Sensitivity): 0.5862068965517241
PR-AUC: 0.49529169775629095
TN: 121, FP: 16, FN: 12, TP: 17



Feed-forward neural network

In [ ]:
pipeline_NN = Pipeline(steps=[
    ('scaler', RobustScaler()),
    ('classifier', MLPClassifier(early_stopping=True, random_state=4)) 
])

param_grid_NN = {
    'classifier__hidden_layer_sizes': [(32, 16, 8), (65,)],
    'classifier__activation': ['relu','tanh'],
    'classifier__solver': ['adam', 'sgd'],
    'classifier__learning_rate_init': [0.001, 0.01],
    'classifier__alpha': [0.0001, 0.01],
    'classifier__learning_rate': ['constant'],
    'classifier__max_iter': [50, 100]
}

param_dist_NN = {
    'classifier__hidden_layer_sizes': [(32, 16, 8), (64,), (128, 64)],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__solver': ['adam', 'sgd', 'lbfgs'],  
    'classifier__learning_rate_init': loguniform(1e-4, 1e-2),
    'classifier__alpha': loguniform(1e-5, 1e-2),
    'classifier__max_iter': randint(150, 500),
}

random_search_NN = RandomizedSearchCV(
    pipeline_NN,
    param_distributions=param_grid_NN,
    n_iter=30,  # number of random combinations
    cv=kf,
    scoring='average_precision',
    n_jobs=-1,
    random_state=4
)

random_search_NN.fit(x_train, y_train)

print('Best parameters found:\n', random_search_NN.best_params_)
print("Best score:", random_search_NN.best_score_)

In [ ]:
NN_model = random_search_NN.best_estimator_ 
y_pred_NN = NN_model.predict(x_test)  
probabilities_NN = NN_model.predict_proba(x_test)